### Benchmark Hailo

In [ ]:
import os
import sys
import time
import csv
import serial
import subprocess
from datetime import datetime

# ==========================================
# 1. PARÁMETROS DEL EXPERIMENTO (CONFIGURABLE)
# ==========================================
EXP_NAME = "yolov11n_max_FPS"

# Rutas en la Raspberry Pi
DATASET_PATH = "/home/USERNAME/Desktop/dataset/test"

# ==========================================
# 2. RUTAS LOCALES (WINDOWS)
# ==========================================
LOCAL_MODEL_PATH = f"models/binaries/{EXP_NAME}.hef"

MASTER_CSV_PATH = "experiments/master_results.csv"
LOCAL_IMAGES_ROOT = f"experiments/images/{EXP_NAME}"

# ==========================================
# 3. CONFIGURACIÓN DE RED Y HARDWARE
# ==========================================
PUERTO_COM = "COM8"
BAUD_RATE = 115200

RPI_USER_IP = "USERNAME@192.168.1.xx"
RPI_WORKSPACE = "/home/USERNAME/Desktop/scripts"
RPI_PYTHON = f"{RPI_WORKSPACE}/.venv/bin/python3"
RPI_BENCH_SCRIPT = f"{RPI_WORKSPACE}/benchmark_hailo.py"

RPI_RESULTS_BASE = "/home/USERNAME/Desktop/results"
RPI_CSV_FILE = f"{RPI_RESULTS_BASE}/benchmark_hailo.csv"
RPI_IMAGE_DIR = f"{RPI_RESULTS_BASE}/{EXP_NAME}"

CONF = 0.42
IOU = 0.45

# ==========================================
# 4. EJECUCIÓN DEL PROCESO
# ==========================================

# Asegurar existencia de directorios locales
os.makedirs(os.path.dirname(MASTER_CSV_PATH), exist_ok=True)
os.makedirs(LOCAL_IMAGES_ROOT, exist_ok=True)

arduino = None
proceso_ssh = None
power_buffer = []
tiempo_inicio = 0

print(f"[INFO] Iniciando orquestador: {EXP_NAME}")

# --- TRANSFERENCIA PREVIA DEL MODELO A LA RPI ---
if not os.path.exists(LOCAL_MODEL_PATH):
    print(f"[ERROR CRÍTICO] No se encuentra el archivo local: {LOCAL_MODEL_PATH}")
    sys.exit(1)

print(f"[INFO] ⬆️ Subiendo modelo {EXP_NAME}.hef a la Raspberry Pi...")
try:
    subprocess.run(["ssh", RPI_USER_IP, f"mkdir -p $(dirname {MODEL_PATH})"], check=True)
    subprocess.run(["scp", LOCAL_MODEL_PATH, f"{RPI_USER_IP}:{MODEL_PATH}"], check=True)
    print("[INFO] ✅ Transferencia del modelo completada.")
except subprocess.CalledProcessError as e:
    print(f"[ERROR] Falló la subida del modelo: {e}")
    sys.exit(1)
# ---------------------------------------------------------

try:
    # Inicialización de hardware de medición
    arduino = serial.Serial(PUERTO_COM, BAUD_RATE, timeout=0.1)
    time.sleep(2.0)
    arduino.write(b"START\n")

    # Construcción del comando remoto
    remote_cmd = f"{RPI_PYTHON} {RPI_BENCH_SCRIPT} --exp_name {EXP_NAME} --model {MODEL_PATH} --dataset {DATASET_PATH} --conf {CONF} --iou {IOU}"
    ssh_cmd = ["ssh", "-o", "BatchMode=yes", "-o", "StrictHostKeyChecking=no", RPI_USER_IP, remote_cmd]

    print(f"[INFO] Ejecutando benchmark en RPi5 e iniciando captura de telemetría...")

    tiempo_inicio = time.time()
    proceso_ssh = subprocess.Popen(ssh_cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)

    last_ui_update = time.time()
    while proceso_ssh.poll() is None:
        if arduino.in_waiting > 0:
            linea = arduino.readline().decode('utf-8', errors='ignore').strip()
            if linea.startswith("DATA,"):
                try:
                    val = linea.split(",")[3] # Potencia (W)
                    potencia = float(val)
                    power_buffer.append(potencia)

                    if time.time() - last_ui_update >= 1.0:
                        print(f"[MONITOR] Potencia: {potencia:.2f}W | Muestras: {len(power_buffer)}", end='\r')
                        last_ui_update = time.time()
                except (ValueError, IndexError):
                    pass

    # Mostrar logs de la RPi si algo va mal o para confirmar
    if proceso_ssh.returncode != 0:
        print(f"\n[ERROR] El benchmark remoto falló (Code {proceso_ssh.returncode})")
        print(proceso_ssh.stderr.read())
    else:
        print("\n[INFO] Benchmark remoto finalizado exitosamente. Salida de la RPi:")
        print("==================================================")
        print(proceso_ssh.stdout.read().strip())
        print("==================================================")

except KeyboardInterrupt:
    print("\n[WARN] Interrupción manual detectada.")
    if proceso_ssh: proceso_ssh.terminate()
finally:
    if arduino and arduino.is_open:
        arduino.write(b"STOP\n")
        arduino.close()

    duracion = time.time() - tiempo_inicio

    # 5. CONSOLIDACIÓN DE DATOS Y TRANSFERENCIA DE ARCHIVOS
    if power_buffer and proceso_ssh and proceso_ssh.returncode == 0:
        avg_p = sum(power_buffer) / len(power_buffer)
        max_p = max(power_buffer)
        energy_kwh = (avg_p * (duracion / 3600)) / 1000

        temp_csv = "temp_fetch.csv"
        try:
            subprocess.run(["scp", f"{RPI_USER_IP}:{RPI_CSV_FILE}", temp_csv], check=True, capture_output=True)

            with open(temp_csv, mode='r') as f:
                data = list(csv.DictReader(f))
                # Búsqueda segura
                record = next((r for r in reversed(data) if r.get("Experiment_ID", "").strip() == EXP_NAME.strip()), None)

            if record:
                record.update({
                    "Mean_Power_W": round(avg_p, 2),
                    "Max_Power_W": round(max_p, 2),
                    "Total_Energy_kWh": round(energy_kwh, 8),
                    "Duration_Secs": round(duracion, 2)
                })

                es_nuevo_o_vacio = not os.path.isfile(MASTER_CSV_PATH) or os.path.getsize(MASTER_CSV_PATH) == 0

                with open(MASTER_CSV_PATH, mode='a', newline='') as f:
                    writer = csv.DictWriter(f, fieldnames=record.keys())

                    if es_nuevo_o_vacio:
                        writer.writeheader() # Escribe cabeceras solo si está vacío

                    writer.writerow(record)
                print(f"[INFO] Registro consolidado en: {MASTER_CSV_PATH}")

                # 6. TRANSFERENCIA DE ARTEFACTOS
                print(f"[INFO] Transfiriendo imágenes del experimento...")
                local_exp_dir = os.path.join(LOCAL_IMAGES_ROOT, EXP_NAME)
                os.makedirs(local_exp_dir, exist_ok=True)

                transfer_cmd = ["scp", "-r", f"{RPI_USER_IP}:{RPI_IMAGE_DIR}/*", local_exp_dir]
                res = subprocess.run(transfer_cmd, capture_output=True)

                if res.returncode == 0:
                    print(f"[INFO] Artefactos visuales guardados en: {local_exp_dir}")
                else:
                    print(f"[WARN] No se pudieron traer imágenes (quizás no generó ninguna): {res.stderr.decode()}")
            else:
                print(f"\n[ERROR] No se encontró 'Experiment_ID' == '{EXP_NAME}' en el CSV.")
                if data:
                    print(f"[DEBUG] Última fila encontrada: {data[-1]}")

        except subprocess.CalledProcessError as e:
            print(f"\n[ERROR] Fallo al descargar el CSV vía SCP: {e.stderr.decode()}")
        finally:
            if os.path.exists(temp_csv): os.remove(temp_csv)
    else:
        print("[WARN] No se generaron resultados finales o no hubo telemetría.")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os

# ==========================================
# 1. CONFIGURACIÓN Y CARGA DE DATOS
# ==========================================
MASTER_CSV_PATH = "experiments/master_results.csv"

if not os.path.exists(MASTER_CSV_PATH):
    print(f"[ERROR] Archivo no encontrado: {MASTER_CSV_PATH}")
else:
    # Carga del dataset
    df = pd.read_csv(MASTER_CSV_PATH)

    # 1. Filtrado
    df = df[df['Experiment_ID'].str.contains('balanced')]
    df['Experiment_ID'] = df['Experiment_ID'].str.replace('_balanced', '', regex=False)

    df['Experiment_ID'] = pd.Categorical(df['Experiment_ID'], categories=['yolov11n', 'yolov11s'], ordered=True)
    df = df.sort_values('Experiment_ID')

    # Creación de métricas derivadas
    df['Efficiency (FPS/W)'] = df['FPS_Inference'] / df['Mean_Power_W']

    # Configuración de estilo global para papers/memorias
    sns.set_theme(style="whitegrid")
    plt.rcParams.update({'font.size': 12, 'font.family': 'sans-serif'})

    # Paleta de colores consistente (Verde para Nano, Morado/Azul para Small)
    custom_palette = {'yolov11n': '#2ecc71', 'yolov11s': '#9b59b6'}

    # Función auxiliar para poner el valor numérico encima de las barras
    def add_value_labels(ax, spacing=5, is_percentage=False):
        for rect in ax.patches:
            y_value = rect.get_height()
            x_value = rect.get_x() + rect.get_width() / 2
            space = spacing
            va = 'bottom'
            if y_value < 0:
                space *= -1
                va = 'top'
            label = f"{y_value:.2f}"
            if is_percentage: label += "%"
            ax.annotate(label, (x_value, y_value), xytext=(0, space),
                        textcoords="offset points", ha='center', va=va, fontweight='bold')

    # ==========================================
    # GRÁFICA 1: Rendimiento Predictivo (F1-Score)
    # ==========================================
    plt.figure(figsize=(8, 6))
    ax1 = sns.barplot(x='Experiment_ID', y='F1_Score', data=df, palette=custom_palette, edgecolor='black')

    plt.title('Precisión Global en el Hardware Final (F1-Score)', fontsize=14, fontweight='bold', pad=15)
    plt.xlabel('Modelo Desplegado (INT8)', fontweight='bold')
    plt.ylabel('F1-Score', fontweight='bold')
    plt.ylim(0, 1.0)
    add_value_labels(ax1)
    plt.tight_layout()
    plt.show()

    # ==========================================
    # GRÁFICA 2: Rendimiento Temporal (FPS)
    # ==========================================
    plt.figure(figsize=(8, 6))
    ax2 = sns.barplot(x='Experiment_ID', y='FPS_Inference', data=df, palette=custom_palette, edgecolor='black')

    plt.title('Velocidad de Inferencia (Latencia Operativa)', fontsize=14, fontweight='bold', pad=15)
    plt.xlabel('Modelo Desplegado', fontweight='bold')
    plt.ylabel('Fotogramas por Segundo (FPS)', fontweight='bold')
    # Añadimos una línea roja indicando el mínimo para "Tiempo Real" (ej. 30 FPS)
    plt.axhline(y=30, color='r', linestyle='--', linewidth=2, label='Objetivo Mínimo (30 FPS)')
    plt.legend()
    add_value_labels(ax2)
    plt.tight_layout()
    plt.show()

    # ==========================================
    # GRÁFICA 3: Consumo Energético (Vatios)
    # ==========================================
    plt.figure(figsize=(8, 6))
    ax3 = sns.barplot(x='Experiment_ID', y='Mean_Power_W', data=df, palette=custom_palette, edgecolor='black')

    plt.title('Consumo de Potencia Medio del Sistema', fontsize=14, fontweight='bold', pad=15)
    plt.xlabel('Modelo Desplegado', fontweight='bold')
    plt.ylabel('Potencia Media (W)', fontweight='bold')
    add_value_labels(ax3)
    plt.tight_layout()
    plt.show()

    # ==========================================
    # GRÁFICA 4: Comportamiento Térmico (°C)
    # ==========================================
    plt.figure(figsize=(8, 6))
    ax4 = sns.barplot(x='Experiment_ID', y='Mean_Temp_C', data=df, palette=custom_palette, edgecolor='black')

    plt.title('Temperatura Media del Coprocesador Hailo-8', fontsize=14, fontweight='bold', pad=15)
    plt.xlabel('Modelo Desplegado', fontweight='bold')
    plt.ylabel('Temperatura (°C)', fontweight='bold')
    # Añadimos una línea de advertencia térmica (Throttling suele empezar a los 65-70°C)
    plt.axhline(y=65, color='orange', linestyle='--', linewidth=2, label='Umbral de Alarma Térmica')
    plt.legend()
    add_value_labels(ax4)
    plt.tight_layout()
    plt.show()

    # ==========================================
    # REPORTE TABULAR CONSOLIDADO PARA LATEX
    # ==========================================
    print("\n" + "="*80)
    print(" TABLA RESUMEN PARA MEMORIA LATEX")
    print("="*80)

    summary_cols = ['Experiment_ID', 'F1_Score', 'FPS_Inference', 'Mean_Power_W', 'Efficiency (FPS/W)', 'Mean_Temp_C', 'Mean_CPU_%', 'Mean_RAM_%']
    df_summary = df[summary_cols].copy()

    # Renombrar columnas para que queden bonitas
    df_summary.columns = ['Modelo', 'F1-Score', 'FPS', 'Potencia (W)', 'Eficiencia (FPS/W)', 'Temp (ºC)', 'CPU (%)', 'RAM (%)']

    pd.options.display.float_format = '{:.2f}'.format
    display(df_summary.reset_index(drop=True))

### Benchmark CPU

In [ ]:
import os
import time
import csv
import serial
import subprocess
from datetime import datetime


# ==========================================
# 1. PARÁMETROS DEL EXPERIMENTO (CONFIGURABLE)
# ==========================================
EXP_NAME = "yolov11s"
MODEL_PATH = f"/home/USERNAME/Desktop/models/yolo/{EXP_NAME}.pt"
DATASET_PATH = "/home/USERNAME/Desktop/dataset/test"

# Umbrales de inferencia
CONFIDENCE_TH = 0.42
IOU_TH = 0.45

# ==========================================
# 2. RUTAS LOCALES (WINDOWS)
# ==========================================
# Ruta absoluta del archivo CSV único maestro
MASTER_CSV_PATH = "experiments/master_yolo_results.csv"

# Directorio raíz local donde se descargarán las imágenes y matrices
LOCAL_IMAGES_ROOT = "experiments/artifacts_yolo/"

# ==========================================
# 3. CONFIGURACIÓN DE RED Y HARDWARE
# ==========================================
PUERTO_COM = "COM8"
BAUD_RATE = 115200

RPI_USER_IP = "USERNAME@192.168.1.xx"
RPI_WORKSPACE = "/home/USERNAME/Desktop/scripts"

# Llamada directa al intérprete virtual
RPI_PYTHON = f"{RPI_WORKSPACE}/.venv/bin/python3"
RPI_BENCH_SCRIPT = f"{RPI_WORKSPACE}/benchmark_yolo.py"

RPI_EXP_DIR = f"/home/USERNAME/Desktop/results_yolo/{EXP_NAME}"
RPI_CSV_FILE = f"{RPI_EXP_DIR}/benchmark.csv"

# ==========================================
# 4. EJECUCIÓN DEL PROCESO
# ==========================================

# Asegurar existencia de directorios locales en Windows
os.makedirs(os.path.dirname(MASTER_CSV_PATH), exist_ok=True)
local_exp_dir = os.path.join(LOCAL_IMAGES_ROOT, EXP_NAME)
os.makedirs(local_exp_dir, exist_ok=True)

arduino = None
proceso_ssh = None
power_buffer = []
tiempo_inicio = 0

print(f"[INFO] Iniciando orquestador para Ultralytics YOLO: {EXP_NAME}")

try:
    # 1. Inicialización de hardware de medición
    print(f"[INFO] Abriendo puerto serial {PUERTO_COM}...")
    arduino = serial.Serial(PUERTO_COM, BAUD_RATE, timeout=0.1)
    time.sleep(2.0)
    arduino.write(b"START\n")

    # 2. Construcción del comando remoto
    remote_cmd = (
        f"{RPI_PYTHON} {RPI_BENCH_SCRIPT} "
        f"--exp_name {EXP_NAME} "
        f"--model {MODEL_PATH} "
        f"--dataset {DATASET_PATH} "
        f"--output {RPI_CSV_FILE} "
        f"--conf {CONFIDENCE_TH} "
        f"--iou {IOU_TH}"
    )

    # 3. Creación del directorio remoto (Con bloqueo de Stdin)
    print(f"[INFO] Preparando directorio de trabajo remoto...")
    mkdir_cmd = ["ssh", "-o", "BatchMode=yes", "-o", "StrictHostKeyChecking=no", RPI_USER_IP, f"mkdir -p {RPI_EXP_DIR}"]
    subprocess.run(mkdir_cmd, check=True, timeout=10, stdin=subprocess.DEVNULL)

    ssh_cmd = ["ssh", "-o", "BatchMode=yes", "-o", "StrictHostKeyChecking=no", RPI_USER_IP, remote_cmd]
    print(f"[INFO] Ejecutando inferencia en RPi5 e iniciando captura de telemetría...")

    tiempo_inicio = time.time()
    out_log = open("yolo_stdout.log", "w")
    err_log = open("yolo_stderr.log", "w")

    proceso_ssh = subprocess.Popen(
        ssh_cmd,
        stdout=out_log,
        stderr=err_log,
        stdin=subprocess.DEVNULL,
        text=True
    )

    last_ui_update = time.time()

    # Bucle de captura de datos seriales
    while proceso_ssh.poll() is None:
        if arduino.in_waiting > 0:
            linea = arduino.readline().decode('utf-8', errors='ignore').strip()
            if linea.startswith("DATA,"):
                try:
                    potencia = float(linea.split(",")[3])
                    power_buffer.append(potencia)

                    if time.time() - last_ui_update >= 1.0:
                        print(f"[MONITOR] Potencia instantánea: {potencia:.2f} W | Muestras: {len(power_buffer)}", end='\r')
                        last_ui_update = time.time()
                except (ValueError, IndexError):
                    pass

    # Verificación del estado de salida del proceso remoto
    if proceso_ssh.returncode != 0:
        print(f"\n[ERROR] El benchmark remoto falló (Código de salida: {proceso_ssh.returncode})")
        print(f"[DETALLE]\n{proceso_ssh.stderr.read()}")
    else:
        print("\n[INFO] Benchmark remoto finalizado exitosamente.")

except KeyboardInterrupt:
    print("\n[WARN] Interrupción manual detectada. Abortando proceso remoto...")
    if proceso_ssh: proceso_ssh.terminate()
except Exception as e:
    print(f"\n[ERROR] Excepción crítica: {e}")

finally:
    # Secuencia de cierre del instrumento de medida
    if arduino and arduino.is_open:
        arduino.write(b"STOP\n")
        arduino.close()

    try:
        out_log.close()
        err_log.close()
    except:
        pass

    duracion = time.time() - tiempo_inicio


    # 5. CONSOLIDACIÓN DE DATOS Y EXTRACCIÓN DE ARTEFACTOS
    if power_buffer and proceso_ssh and proceso_ssh.returncode == 0:
        avg_power = sum(power_buffer) / len(power_buffer)
        max_power = max(power_buffer)
        energy_kwh = (avg_power * (duracion / 3600)) / 1000

        print(f"[INFO] Consumo medio: {avg_power:.2f} W | Máximo: {max_power:.2f} W")

        temp_csv = "temp_yolo_fetch.csv"

        try:
            # Recuperación del CSV generado en la iteración actual
            subprocess.run(["scp", f"{RPI_USER_IP}:{RPI_CSV_FILE}", temp_csv], check=True, capture_output=True)

            with open(temp_csv, mode='r') as f:
                data = list(csv.DictReader(f))
                # Se asume que el script remoto puede haber inicializado el CSV; tomamos el último registro que coincida
                record = next((r for r in reversed(data) if r["Experiment_ID"] == EXP_NAME), None)

            if record:
                # Inyección de métricas telemétricas
                record.update({
                    "Mean_Power_W": round(avg_power, 2),
                    "Max_Power_W": round(max_power, 2),
                    "Total_Energy_kWh": round(energy_kwh, 8),
                    "Duration_Secs": round(duracion, 2)
                })

                # Persistencia en el archivo maestro
                file_exists = os.path.isfile(MASTER_CSV_PATH)
                with open(MASTER_CSV_PATH, mode='a', newline='') as f:
                    writer = csv.DictWriter(f, fieldnames=record.keys())
                    if not file_exists: writer.writeheader()
                    writer.writerow(record)
                print(f"[INFO] Datos telemétricos consolidados en registro maestro: {MASTER_CSV_PATH}")

                # Extracción de artefactos visuales (Imágenes y Matriz de Confusión)
                print(f"[INFO] Sincronizando artefactos visuales...")
                transfer_cmd = ["scp", f"{RPI_USER_IP}:{RPI_EXP_DIR}/*.png", local_exp_dir]
                transfer_cmd_jpg = ["scp", f"{RPI_USER_IP}:{RPI_EXP_DIR}/*.jpg", local_exp_dir]

                # Se ejecutan ambos por si las extensiones varían, omitiendo errores si no hay de un tipo
                subprocess.run(transfer_cmd, stderr=subprocess.DEVNULL)
                subprocess.run(transfer_cmd_jpg, stderr=subprocess.DEVNULL)

                print(f"[INFO] Artefactos transferidos al directorio local: {local_exp_dir}")

            if os.path.exists(temp_csv): os.remove(temp_csv)

        except subprocess.CalledProcessError as e:
            print(f"[ERROR] Fallo durante la sincronización SCP: {e}")
    else:
        print("[WARN] Los resultados no pudieron ser consolidados.")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Rutas a tus archivos CSV
HAILO_CSV = "experiments/master_results.csv"
CPU_CSV = "experiments/master_yolo_results.csv"

def load_and_merge_data():
    # Cargar datos
    df_hailo = pd.read_csv(HAILO_CSV)
    df_cpu = pd.read_csv(CPU_CSV)

    # Filtrar solo estrategias balanced del Hailo (si fuera necesario) y limpiar nombres
    df_hailo = df_hailo[df_hailo['Experiment_ID'].str.contains('balanced')].copy()
    df_hailo['Model_Base'] = df_hailo['Experiment_ID'].str.replace('_balanced', '')
    df_cpu['Model_Base'] = df_cpu['Experiment_ID']

    # Añadir columna identificativa
    df_hailo['Hardware'] = 'NPU (Hailo-8)'
    df_cpu['Hardware'] = 'CPU (ARM Cortex-A76)'

    # Unir ambos DataFrames
    df_merged = pd.concat([df_hailo, df_cpu], ignore_index=True)

    # Forzar el orden de los modelos (yolov11n primero) y el hardware
    df_merged['Model_Base'] = pd.Categorical(df_merged['Model_Base'], categories=['yolov11n', 'yolov11s'], ordered=True)
    df_merged['Hardware'] = pd.Categorical(df_merged['Hardware'], categories=['CPU (ARM Cortex-A76)', 'NPU (Hailo-8)'], ordered=True)
    df_merged = df_merged.sort_values(['Model_Base', 'Hardware'])

    # Calcular Eficiencia Energética (FPS / Vatio)
    df_merged['Eficiencia_FPS_W'] = df_merged['FPS_Inference'] / df_merged['Mean_Power_W']

    return df_merged

def plot_thesis_benchmarks_separated(df):
    sns.set_theme(style="whitegrid", context="paper", font_scale=1.2)
    plt.rcParams.update({'font.family': 'sans-serif'})

    # Paleta de colores corporativa: Rojo (CPU) vs Verde (Hailo)
    palette = {'CPU (ARM Cortex-A76)': '#E63946', 'NPU (Hailo-8)': '#2A9D8F'}

    def add_value_labels(ax, format_str='%.2f', is_f1=False):
        for container in ax.containers:
            labels = [format_str % v if v > 0 else "" for v in container.datavalues]
            ax.bar_label(container, labels=labels, padding=3, fontweight='bold')

    # ==========================================
    # Gráfica 1: Precisión (F1-Score)
    # ==========================================
    plt.figure(figsize=(8, 6))
    ax1 = sns.barplot(data=df, x='Model_Base', y='F1_Score', hue='Hardware', palette=palette, edgecolor='black')
    plt.title('Precisión Predictiva: FP32 (CPU) vs INT8 (NPU)', fontweight='bold', pad=15)
    plt.ylabel('F1 Score')
    plt.xlabel('Modelo Evaluado')
    plt.ylim(0.8, 0.95) # Zoom para ver diferencias
    add_value_labels(ax1, '%.3f')
    plt.legend(title='Arquitectura de Ejecución')
    plt.tight_layout()
    plt.show()

    # ==========================================
    # Gráfica 2: Rendimiento (FPS)
    # ==========================================
    plt.figure(figsize=(8, 6))
    ax2 = sns.barplot(data=df, x='Model_Base', y='FPS_Inference', hue='Hardware', palette=palette, edgecolor='black')
    plt.title('Latencia Operativa: Fotogramas por Segundo (FPS)', fontweight='bold', pad=15)
    plt.ylabel('FPS')
    plt.xlabel('Modelo Evaluado')
    plt.axhline(y=30, color='orange', linestyle='--', label='Tiempo Real (30 FPS)')
    add_value_labels(ax2, '%.1f')
    plt.legend(title='Arquitectura de Ejecución')
    plt.tight_layout()
    plt.show()

    # ==========================================
    # Gráfica 3: Consumo de Potencia (W)
    # ==========================================
    plt.figure(figsize=(8, 6))
    ax3 = sns.barplot(data=df, x='Model_Base', y='Mean_Power_W', hue='Hardware', palette=palette, edgecolor='black')
    plt.title('Consumo de Potencia Eléctrica Global del Sistema', fontweight='bold', pad=15)
    plt.ylabel('Potencia Media (W)')
    plt.xlabel('Modelo Evaluado')
    add_value_labels(ax3, '%.2f')
    plt.legend(title='Arquitectura de Ejecución')
    plt.tight_layout()
    plt.show()

    # ==========================================
    # Gráfica 4: Temperatura (ºC)
    # ==========================================
    plt.figure(figsize=(8, 6))
    ax4 = sns.barplot(data=df, x='Model_Base', y='Mean_Temp_C', hue='Hardware', palette=palette, edgecolor='black')
    plt.title('Estrés Térmico del Sistema', fontweight='bold', pad=15)
    plt.ylabel('Temperatura (ºC)')
    plt.xlabel('Modelo Evaluado')
    plt.axhline(y=65, color='red', linestyle='--', label='Riesgo Throttling (65ºC)')
    add_value_labels(ax4, '%.1f')
    plt.legend(title='Arquitectura de Ejecución')
    plt.tight_layout()
    plt.show()

try:
    df_results = load_and_merge_data()
    plot_thesis_benchmarks_separated(df_results)
except FileNotFoundError as e:
    print(f"❌ Error al cargar los archivos: {e}")